# 07 · Evaluation

**Stage 7 of 8** · ACCESS → WRANGLE → EXPLORE → VISUALIZE → ENGINEER → MODEL → EVALUATE → RECOMMEND

> **Project premise:** if someone has had cancer and was treated, **will it come back?** And for every factor linked to it coming back: **correlation or causation?**
>
> All data here is public: a k-anonymous feature table (every combination of site group, region, sex and diagnosis period is shared by ≥ 3 patients), synthetic patients, and aggregate reports. The raw database and clinical notes stay private.

**Goal of this stage:** decide whether the model is reliable enough to support a follow-up decision, using more than accuracy.

**Business question:** If the clinic flags patients for early follow-up using this model, how many relapses would it catch, and at what cost?

In [1]:
%matplotlib inline
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
PUBLIC = ROOT / "data" / "public" / "patients_features_public.csv"
SYNTH = ROOT / "data" / "synthetic" / "synthetic_patients.csv"
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8984"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.axisbelow": True, "figure.dpi": 110})
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
df = pd.read_csv(PUBLIC)

In [2]:
import model as M
X, y = M.load(PUBLIC)
res, summary, oof, _ = M.cross_validate(X, y)
thr = y.mean()

### Q1 · Accuracy

In [3]:
summary.xs("mean", axis=1, level=1)[["accuracy"]].round(3)

,accuracy
model,
Baseline (random flagging),0.648
Logistic regression,0.646
Random forest,0.728


Accuracy looks fine (~0.65–0.73), but **always predicting "no relapse" would already score 78%**, because only 22% relapse.
That's why accuracy alone is misleading here.

### Q2 · Confusion matrix (random forest, averaged over the 20 repeats)

In [4]:
from sklearn.metrics import confusion_matrix
rf = oof["Random forest"]
cm = np.mean([confusion_matrix(y, (rf[r] >= np.quantile(rf[r], 1 - thr)).astype(int)) for r in range(M.N_REPEATS)], axis=0)
pd.DataFrame(cm.round(1), index=["actual: no relapse", "actual: relapse"],
             columns=["predicted: no", "predicted: relapse"])

,predicted: no,predicted: relapse
actual: no relapse,89.2,18.8
actual: relapse,18.8,11.2


### Q3–Q5 · Precision, recall, F1

In [5]:
summary.xs("mean", axis=1, level=1)[["precision", "recall", "f1", "roc_auc"]].round(3)

,precision,recall,f1,roc_auc
model,,,,
Baseline (random flagging),0.190,0.190,0.190,0.492
Logistic regression,0.187,0.187,0.187,0.475
Random forest,0.375,0.375,0.375,0.588


| Metric | Technical meaning | Clinic meaning |
|---|---|---|
| Accuracy | share of correct predictions | how often the flag is right overall |
| **Precision** | correct among flagged | of patients we bring back early, how many really relapse? |
| **Recall** | found among actual relapses | of patients whose cancer will come back, how many do we catch? |
| F1 | balance of the two | coverage vs wasted visits |
| False negative | missed relapse | cancer found later, when it may be harder to treat |
| False positive | flagged, no relapse | an extra visit or scan |

### Q6 · Operating points: how many patients should we flag?

In [6]:
ops = M.operating_points(y.values, oof)
ops.pivot(index="flag_top_share", columns="model", values="recall").style.format("{:.0%}") \
   .set_caption("Recall (share of relapses caught) when flagging the top X% riskiest")

model,Baseline (random flagging),Logistic regression,Random forest
flag_top_share,,,
0.200000,18%,17%,36%
0.360000,34%,36%,46%
0.500000,47%,49%,56%
0.700000,70%,69%,75%


### Q7 · Which error matters more?
A **false negative** (a relapse we miss) costs more than a **false positive** (an extra visit), so for follow-up
planning, **recall matters more than precision**. Even so, at every flagging level the models catch only a little
more than random flagging. This model is **not reliable enough** to drive follow-up decisions on its own.

### Time-aware evaluation (the primary metric)

In [7]:
pd.DataFrame(M.cox_cv(PUBLIC)).T

,c_index_mean,c_index_sd
Cox: site + grade + treatments,0.629,0.027
Cox: breast vs other only,0.593,0.024


The **C-index** asks: of two patients, does the model give higher risk to the one who relapses first? 0.5 = chance,
1.0 = perfect. A cross-validated ~0.63 means **weak** discrimination.

## So what?

Evaluated honestly (cross-validated, no leakage, compared with a random baseline), the model is **weak**. The
right action is not to deploy it, but to **collect the data that would make it work**: stage, age, receptor status, and
dates of relapse and death.